In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ==============================================================================
# STEP 2: DATA LOADING (Wave 4 - 2011)
# ==============================================================================
# ใช้ convert_categoricals=False เพื่อป้องกัน Error จาก Duplicate Labels
file_2011 = 'wave-4-shocksclean.dta'
df_11 = pd.read_stata(file_2011, convert_categoricals=False)

print(f"Loaded Wave 4 (2011) successfully: {len(df_11)} rows")

# ==============================================================================
# STEP 3: DATA CLEANING (Standard Research Logic)
# ==============================================================================
def clean_wave4(df):
    # จัดการ Missing Values ตามมาตรฐาน Stata
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # จัดการคอลัมน์การเงิน (Loss Amount: _x31005a)
    # ปี 2011 อาจมี 'none' หรือข้อมูลประเภท string ปนอยู่
    if '_x31005a' in df.columns:
        df['_x31005a'] = df['_x31005a'].astype(str).str.replace('none', '0', case=False).str.strip()
        df['_x31005a'] = pd.to_numeric(df['_x31005a'], errors='coerce').fillna(0)
    
    return df

df_11 = clean_wave4(df_11)

# ==============================================================================
# STEP 4: COPING STRATEGY HARMONIZATION (เจาะลึก _x31008 - _x31010)
# ==============================================================================
# Mapping รหัสรับมือปี 2011 ให้สอดคล้องกับ Master Template
coping_map_11 = {
    11: 'sold_assets', 12: 'sold_assets', 13: 'sold_assets', 14: 'sold_assets',
    15: 'used_savings', 16: 'used_insurance',
    17: 'borrowed_informal', 18: 'borrowed_informal',
    21: 'borrowed_formal', 22: 'borrowed_formal', 23: 'borrowed_formal',
    28: 'gov_help', 29: 'gov_help', 30: 'relatives_help'
}

# สร้างคอลัมน์ Binary (0/1)
coping_cols = ['sold_assets', 'used_savings', 'used_insurance', 'borrowed_informal', 'borrowed_formal', 'gov_help']
for c in coping_cols:
    df_11[f'coping_{c}'] = 0

# วนลูปตรวจสอบจากลำดับการรับมือทั้ง 3 ช่อง
for col in ['_x31008', '_x31009', '_x31010']:
    if col in df_11.columns:
        for code, name in coping_map_11.items():
            df_11.loc[df_11[col] == code, f'coping_{name}'] = 1

# ==============================================================================
# STEP 5: SHOCK GROUPING (Categorization)
# ==============================================================================
# ใช้รหัสเดียวกับมาตรฐานงานวิจัย (Harmonization)
shock_map_11 = {
    10: 'agricultural', 11: 'agricultural', 63: 'agricultural', 55: 'agricultural',
    1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
    5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
    8: 'social', 70: 'social', 77: 'economics'
}

df_11['shocks_Group'] = df_11['_x31002'].map(shock_map_11).fillna('others')
df_11['survey_year'] = 2011

# ==============================================================================
# STEP 6: EXPORT
# ==============================================================================
output_file = 'shocks_2011_cleaned_final.csv'
df_11.to_csv(output_file, index=False)

# สรุปภาพรวมด้วยกราฟ
sns.countplot(data=df_11, x='shocks_Group', palette='rocket')
plt.title('Shock Distribution (Wave 4 - 2011)')
plt.show()

print(f"✅ Process Completed. File saved as {output_file}")